In [1]:
import os 
import json 
import os 
import json 
from tqdm import tqdm 

path = "../../data/sciERC"

def read_ann(path, filename):
    entities = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Only entity lines start with T
            if not line.startswith("T"):
                continue

            tid, meta, text = line.split("\t")

            label, start, end = meta.split()
            start = int(start)
            end = int(end)

            entities.append({
                "filename": filename,
                "id": tid,
                "label": label,
                "start": start,
                "end": end,
                "text": text,
            })

    return entities


path = "../../data/sciERC"
total_ann_data = []
for file in os.listdir(path):
    if file.endswith(".ann"):
        file_path = os.path.join(path, file)
        doc_name = file.split('.ann')[0]
        total_ann_data.append(read_ann(file_path, doc_name))

In [2]:
labels = ["Material", "Method", "Task",]

filtered = []
for ann_sample in tqdm(total_ann_data):
    for span in ann_sample:
        if span['label'] not in labels:
            continue
        
        for file in os.listdir(path):
            if file.endswith(".txt"):
                doc_name = file.split('.txt')[0]
                if span['filename'] == doc_name:
                    file_path = os.path.join(path, file)
                    with open(file_path, 'r') as f:
                        full_text = f.read()
                    filtered.append(span | {"document": full_text})

  0%|          | 0/500 [00:00<?, ?it/s]

100%|██████████| 500/500 [00:07<00:00, 68.17it/s]


In [3]:
import re
from collections import defaultdict
from typing import List, Dict, Any, Tuple

# ----------------------------
# Tokenization (simple + offset-preserving)
# ----------------------------
_TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize_with_offsets(text: str) -> List[Tuple[str, int, int]]:
    """Returns [(token, start_char, end_char), ...] where end_char is exclusive."""
    return [(m.group(0), m.start(), m.end()) for m in _TOKEN_RE.finditer(text)]


# ----------------------------
# BIO labeling from character spans (records share same doc)
# ----------------------------
def ann_records_to_bio_one_doc(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    records: list of dicts like
      {"filename":..., "label":..., "start":..., "end":..., "document":...}

    returns:
      {"filename":..., "text":..., "tokens":[...], "labels":[...]}
    """
    if not records:
        return {"filename": "", "text": "", "tokens": [], "labels": []}

    filename = str(records[0].get("filename", ""))
    text = str(records[0].get("document", "") or "")

    toks = tokenize_with_offsets(text)
    tokens = [t for t, _, _ in toks]
    labels = ["O"] * len(tokens)

    # normalize spans
    spans = []
    for r in records:
        if r.get("start") is None or r.get("end") is None:
            continue
        start = int(r["start"])
        end = int(r["end"])
        if end <= start:
            continue
        lab = str(r.get("label", "")).strip()
        if not lab:
            continue
        spans.append({"label": lab, "start": start, "end": end})

    # earlier start first; if tie, longer span first
    spans.sort(key=lambda x: (x["start"], -(x["end"] - x["start"])))

    def overlaps(tok_s: int, tok_e: int, sp_s: int, sp_e: int) -> bool:
        return tok_s < sp_e and tok_e > sp_s

    # assign BIO (do not overwrite already-labeled tokens)
    for sp in spans:
        sp_s, sp_e, lab = sp["start"], sp["end"], sp["label"]
        idxs_all = [i for i, (_, ts, te) in enumerate(toks) if overlaps(ts, te, sp_s, sp_e)]
        if not idxs_all:
            continue

        idxs = [i for i in idxs_all if labels[i] == "O"]
        if not idxs:
            continue

        labels[idxs[0]] = f"B-{lab}"
        for i in idxs[1:]:
            labels[i] = f"I-{lab}"

    return {"filename": filename, "text": text, "tokens": tokens, "labels": labels}


# ----------------------------
# NEW: convert a single flat list of dicts -> list of BIO docs
# ----------------------------
def convert_flat_records_to_bio(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Input: flat list of annotation dicts (possibly many files/docs mixed)
    Output: list of BIO examples, one per (filename, document) group.
    """
    if not isinstance(records, list):
        raise TypeError("records must be a list of dictionaries")

    groups = defaultdict(list)

    for r in records:
        if not isinstance(r, dict):
            continue
        fn = str(r.get("filename", ""))
        doc = str(r.get("document", "") or "")
        if not fn and not doc:
            # skip unusable rows
            continue
        groups[(fn, doc)].append(r)

    out = []
    for (fn, doc_text), recs in groups.items():
        # enforce consistency for the per-doc converter
        recs_norm = []
        for r in recs:
            rr = dict(r)
            rr["filename"] = fn
            rr["document"] = doc_text
            recs_norm.append(rr)

        out.append(ann_records_to_bio_one_doc(recs_norm))

    return out

In [4]:

bio_docs = convert_flat_records_to_bio(filtered)
print("Num BIO docs:", len(bio_docs))
print(bio_docs[0]["filename"])
print(list(zip(bio_docs[0]["tokens"], bio_docs[0]["labels"]))[:40])

Num BIO docs: 499
AAAI_2008_262_abs
[('We', 'O'), ('describe', 'O'), ('Yoopick', 'B-Method'), (',', 'O'), ('a', 'O'), ('combinatorial', 'B-Method'), ('sports', 'I-Method'), ('prediction', 'I-Method'), ('market', 'I-Method'), ('that', 'O'), ('implements', 'O'), ('a', 'O'), ('flexible', 'O'), ('betting', 'O'), ('language', 'O'), (',', 'O'), ('and', 'O'), ('in', 'O'), ('turn', 'O'), ('facilitates', 'O'), ('fine', 'B-Task'), ('-', 'I-Task'), ('grained', 'I-Task'), ('probabilistic', 'I-Task'), ('estimation', 'I-Task'), ('of', 'I-Task'), ('outcomes', 'I-Task'), ('.', 'O')]


In [5]:
from sklearn.model_selection import train_test_split

train_data, eval_data = train_test_split(
        bio_docs,
        test_size=0.8,
        random_state=42,
        shuffle=True,
    )
train_path = "../../data/unsupported_claim/sciERC_train.json"
eval_path = "../../data/unsupported_claim/sciERC_eval.json"

with open(train_path, "w", encoding="utf-8") as f:
    json.dump(train_data, f, indent=2, ensure_ascii=False)

with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_data, f, indent=2, ensure_ascii=False)